# UD3.04 — Pandas: GroupBy, combinacion de tablas y series temporales

**Modulo 5073 · Programacion de Inteligencia Artificial · Curso 2026/27**
UD3 — NumPy y Pandas · 14 horas

Criterios 2.b y 2.c · Material de partida de las practicas P3.1 y P3.2


## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Dominar GroupBy** para análisis agregados complejos siguiendo el patrón split-apply-combine
- **Combinar DataFrames** usando merge, join y concatenación para integrar múltiples fuentes de datos
- **Manipular series temporales** con resampling, rolling windows y análisis de tendencias
- **Crear pivot tables** para reorganizar y resumir datos de forma intuitiva
- **Generar visualizaciones** directamente desde Pandas para análisis exploratorio rápido
- **Aplicar todo lo aprendido** en un caso de uso completo de análisis de datos reales

## Introducción

### Del Pandas Básico al Análisis Avanzado

En el notebook anterior aprendimos los fundamentos de Pandas: Series, DataFrames, indexación, limpieza de datos. Ahora vamos a aplicar técnicas avanzadas que son el pan de cada día en Data Science:

**Analogía del mundo real:** Si Pandas básico es como saber usar Excel, Pandas avanzado es como ser un analista experto que puede extraer insights complejos de datasets masivos, combinando múltiples fuentes y detectando tendencias temporales.

### ¿Por qué estas técnicas son cruciales?

1. **GroupBy:** El 80% del análisis de datos implica agrupar y agregar
   - "¿Cuáles son las ventas por región y producto?"
   - "¿Qué departamento tiene el salario promedio más alto?"

2. **Merge/Join:** Los datos reales están distribuidos en múltiples tablas
   - Clientes en una tabla, pedidos en otra, productos en otra
   - Necesitas combinarlas como en SQL

3. **Series Temporales:** Datos con marcas de tiempo están en todas partes
   - Precios de acciones, logs de servidores, datos de sensores IoT
   - Análisis de tendencias, estacionalidad, predicción

4. **Pivot Tables:** Reorganizar datos para análisis multidimensional
   - Crear tablas dinámicas como en Excel, pero programáticamente
   - Resumir grandes volúmenes de datos en vistas comprensibles

### Configuración Inicial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# La visualizacion se estudia a fondo en la UD4. Aqui los graficos son solo
# una herramienta para mirar los datos: nada de estilos ni de bibliotecas extra.
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("numpy", np.__version__, "\u00b7 pandas", pd.__version__)


## 1. GroupBy: Dividir-Aplicar-Combinar

### El Patrón Split-Apply-Combine

GroupBy implementa uno de los patrones más poderosos en análisis de datos. Es **LA** operación que más vas a usar en análisis real.

**Analogía del mundo real - Organizar una clase de estudiantes:**

Imagina que tienes una lista de estudiantes con sus notas:

```
Estudiante Curso Nota
Ana Mate 8.5
Luis Fís 7.2
Carlos Mate 9.1
María Fís 6.8
Pedro Mate 8.9
```

**Pregunta:** ¿Cuál es la nota promedio por curso?

**Con bucles (la forma DIFÍCIL):**
```python
notas_mate = []
notas_fis = []
for estudiante in datos:
    if estudiante.curso == "Mate":
        notas_mate.append(estudiante.nota)
    elif estudiante.curso == "Fís":
        notas_fis.append(estudiante.nota)
        
promedio_mate = sum(notas_mate) / len(notas_mate)
promedio_fis = sum(notas_fis) / len(notas_fis)
```
✗ Mucho código, difícil de mantener, propenso a errores

**Con GroupBy (la forma FÁCIL):**
```python
df.groupby('Curso')['Nota'].mean()
```
✓ Una línea, claro, elegante

### Las 3 Fases de GroupBy (Split-Apply-Combine)

```
DATOS ORIGINALES:
┌──────────┬───────┬──────┐
│Estudiante│ Curso │ Nota │
├──────────┼───────┼──────┤
│ Ana │ Mate │ 8.5 │
│ Luis │ Fís │ 7.2 │
│ Carlos │ Mate │ 9.1 │
│ María │ Fís │ 6.8 │
│ Pedro │ Mate │ 8.9 │
└──────────┴───────┴──────┘
          ↓
   FASE 1: SPLIT (Dividir por grupos)
          ↓
    ┌─────────────┐ ┌─────────────┐
    │ Grupo: Mate │ │ Grupo: Fís │
    ├─────────────┤ ├─────────────┤
    │ Ana 8.5 │ │ Luis 7.2 │
    │ Carlos 9.1 │ │ María 6.8 │
    │ Pedro 8.9 │ └─────────────┘
    └─────────────┘
          ↓ ↓
   FASE 2: APPLY (Aplicar función a cada grupo)
          ↓ ↓
      mean(8.5, 9.1, 8.9) mean(7.2, 6.8)
      = 8.83 = 7.0
          ↓ ↓
   FASE 3: COMBINE (Combinar resultados)
          ↓
┌───────┬──────────┐
│ Curso │ Promedio │
├───────┼──────────┤
│ Mate │ 8.83 │
│ Fís │ 7.00 │
└───────┴──────────┘
```

### ¿Por qué GroupBy es tan Potente?

**1. Responde preguntas de negocio directamente:**
```python
# ¿Cuánto vendemos por región?
df.groupby('Región')['Ventas'].sum()

# ¿Qué producto tiene mayor margen por categoría?
df.groupby('Categoría')['Margen'].max()

# ¿Cuántos clientes tenemos por ciudad?
df.groupby('Ciudad')['ClienteID'].nunique()
```

**2. Evita bucles complejos:**
- Sin GroupBy: 15-20 líneas de código con bucles anidados
- Con GroupBy: 1 línea elegante y rápida

**3. Es extremadamente eficiente:**
- Optimizado internamente en C
- Mucho más rápido que bucles Python
- Puede paralelizarse automáticamente

**4. Flexible:**
- Múltiples columnas de agrupación
- Múltiples funciones de agregación
- Funciones personalizadas

### Sintaxis Básica de GroupBy

```python
# Patrón general:
df.groupby(columnas_para_agrupar)[columnas_a_agregar].funcion_agregacion()
    ↑ ↑ ↑ ↑
    | | | |
    objeto por qué sobre qué columnas qué calcular
    DataFrame agrupar aplicar (sum, mean, etc)
```

### Funciones de Agregación Comunes

| Función | Qué hace | Ejemplo de uso |
|---------|----------|----------------|
| `.sum()` | Suma total | Ventas totales por región |
| `.mean()` | Promedio | Salario promedio por departamento |
| `.median()` | Mediana | Edad mediana por ciudad |
| `.count()` | Contar filas | Número de transacciones por cliente |
| `.nunique()` | Contar valores únicos | Productos diferentes por tienda |
| `.min()` | Valor mínimo | Precio más bajo por categoría |
| `.max()` | Valor máximo | Temperatura máxima por mes |
| `.std()` | Desviación estándar | Variabilidad de notas por curso |
| `.first()` | Primer valor | Fecha de primer pedido por cliente |
| `.last()` | Último valor | Última compra por cliente |

### Cuándo Usar GroupBy

✓ **Usa GroupBy cuando preguntas:**
- "¿Cuánto/cuántos ... **por** ...?"
  - ¿Cuántas ventas **por** región?
  - ¿Cuánto gasta cada cliente **por** mes?
  
- "¿Cuál es el promedio/máximo/mínimo de ... **por** ...?"
  - ¿Cuál es el salario promedio **por** departamento?
  - ¿Cuál es la nota máxima **por** asignatura?

- "¿Cómo se distribuye ... **entre** ...?"
  - ¿Cómo se distribuyen las ventas **entre** categorías?
  
✗ **No uses GroupBy cuando:**
- Solo necesitas filtrar (usa máscaras booleanas)
- Quieres aplicar una función a TODO el DataFrame sin agrupar
- Los cálculos no dependen de grupos

### GroupBy Básico

In [ ]:
# Crear DataFrame de ejemplo: ventas de productos
df_ventas = pd.DataFrame({
    'Producto': ['Laptop', 'Mouse', 'Laptop', 'Teclado', 'Mouse', 'Laptop', 'Teclado', 'Mouse'],
    'Región': ['Norte', 'Norte', 'Sur', 'Norte', 'Sur', 'Este', 'Sur', 'Este'],
    'Vendedor': ['Ana', 'Luis', 'Ana', 'Carlos', 'Luis', 'María', 'Carlos', 'María'],
    'Cantidad': [2, 10, 1, 5, 8, 3, 4, 12],
    'Precio': [899, 25, 899, 75, 25, 899, 75, 25]
})

# Calcular total de ventas
df_ventas['Total'] = df_ventas['Cantidad'] * df_ventas['Precio']

print("DataFrame de ventas:")
print(df_ventas)
print()

In [ ]:
# GroupBy simple: Total de ventas por producto
ventas_por_producto = df_ventas.groupby('Producto')['Total'].sum()

print("Total de ventas por producto:")
print(ventas_por_producto)
print(f"Tipo: {type(ventas_por_producto)}")
print()

In [ ]:
# GroupBy con múltiples agregaciones
resumen_producto = df_ventas.groupby('Producto')['Total'].agg([
    ('Total', 'sum'),
    ('Promedio', 'mean'),
    ('Transacciones', 'count'),
    ('Máximo', 'max'),
    ('Mínimo', 'min')
])

print("Resumen completo por producto:")
print(resumen_producto)
print()

### GroupBy por Múltiples Columnas

In [ ]:
# Agrupar por Producto Y Región
ventas_producto_region = df_ventas.groupby(['Producto', 'Región'])['Total'].sum()

print("Ventas por Producto y Región (MultiIndex):")
print(ventas_producto_region)
print()

# Convertir a DataFrame para mejor visualización
print("Como DataFrame:")
print(ventas_producto_region.reset_index())

In [ ]:
# Agregaciones diferentes para diferentes columnas
resumen_completo = df_ventas.groupby(['Producto', 'Región']).agg({
    'Cantidad': 'sum', # Suma de cantidades
    'Total': ['sum', 'mean'], # Suma y promedio de totales
    'Vendedor': 'count' # Número de transacciones
})

print("Resumen con diferentes agregaciones por columna:")
print(resumen_completo)

### Funciones Personalizadas con GroupBy

In [ ]:
# Definir función personalizada
def rango(series: pd.Series) -> float:
    """Calcula el rango (max - min) de una serie."""
    return series.max() - series.min()

# Aplicar función personalizada
rango_por_producto = df_ventas.groupby('Producto')['Total'].agg([
    ('Total', 'sum'),
    ('Promedio', 'mean'),
    ('Rango', rango),
    ('Desv_Std', 'std')
])

print("Usando función personalizada:")
print(rango_por_producto)

In [ ]:
# transform(): Devuelve un array del mismo tamaño que el original
# Útil para normalizar dentro de grupos

# Calcular el % de contribución de cada venta al total del producto
df_ventas['Contribucion_%'] = (
    df_ventas.groupby('Producto')['Total'].transform(lambda x: x / x.sum() * 100)
)

print("DataFrame con % de contribución:")
print(df_ventas[['Producto', 'Total', 'Contribucion_%']])

### filter(): Filtrar Grupos Completos

In [ ]:
# Mantener solo productos con ventas totales > 1000
productos_alta_venta = df_ventas.groupby('Producto').filter(
    lambda x: x['Total'].sum() > 1000
)

print("Productos con ventas totales > 1000:")
print(productos_alta_venta)
print()
print(f"Productos únicos: {productos_alta_venta['Producto'].unique()}")

## 2. Merge, Join y Concatenación

### ¿Por qué Combinar DataFrames?

En el mundo real, los datos NUNCA vienen en una sola tabla. Están **normalizados** (distribuidos en múltiples tablas) para evitar redundancia.

**Analogía del mundo real - Biblioteca:**

Imagina una biblioteca con dos sistemas:

**Sistema 1: Registro de Préstamos**
```
PréstamoID | UsuarioID | LibroID | Fecha
1001 | U42 | L789 | 2024-01-15
1002 | U15 | L234 | 2024-01-16
```

**Sistema 2: Información de Usuarios**
```
UsuarioID | Nombre | Email
U42 | Ana García | ana@email.com
U15 | Luis Pérez | luis@email.com
```

**Pregunta:** ¿Qué libros prestó Ana García?

**Necesitas COMBINAR ambas tablas** para relacionar préstamos con nombres.

### La Normalización de Bases de Datos

**¿Por qué los datos están separados?**

✗ **MAL: Todo en una tabla (redundancia)**
```
PréstamoID | LibroID | UsuarioNombre | Email | Teléfono
1001 | L789 | Ana García | ana@email.com | 600123456
1002 | L234 | Ana García | ana@email.com | 600123456 ← Duplicado
1003 | L456 | Ana García | ana@email.com | 600123456 ← Duplicado
```
Problemas:
- Duplicación de datos (nombre, email repetidos)
- Si Ana cambia email, hay que actualizar TODAS las filas
- Desperdicio de espacio
- Inconsistencias si actualizas unas filas sí y otras no

✓ **BIEN: Datos normalizados (tablas separadas)**
```
Tabla Préstamos:
PréstamoID | UsuarioID | LibroID
1001 | U42 | L789
1002 | U42 | L234
1003 | U42 | L456

Tabla Usuarios:
UsuarioID | Nombre | Email | Teléfono
U42 | Ana García | ana@email.com | 600123456 ← Una sola vez
```
Ventajas:
- Sin redundancia
- Actualizaciones centralizadas
- Menos espacio
- Datos consistentes

### Tipos de Join (SQL y Pandas)

Pandas soporta los mismos tipos de joins que SQL. La diferencia está en **qué filas se mantienen**:

```
Tabla Izquierda (Pedidos): Tabla Derecha (Clientes):
┌─────────┬────────────┐ ┌────────────┬─────────┐
│PedidoID │ ClienteID │ │ ClienteID │ Nombre │
├─────────┼────────────┤ ├────────────┼─────────┤
│ 101 │ 1 │ │ 1 │ Ana │
│ 102 │ 2 │ │ 2 │ Luis │
│ 103 │ 1 │ │ 3 │ Carlos │
│ 104 │ 5 │ └────────────┴─────────┘
└─────────┴────────────┘
        ↑ ↑
  Cliente 5 NO Cliente 3 NO
  existe en tiene pedidos
  Clientes
```

#### 1. INNER JOIN (how='inner')
**"Solo las filas que coinciden en AMBAS tablas"**

```
Resultado:
┌─────────┬────────────┬─────────┐
│PedidoID │ ClienteID │ Nombre │
├─────────┼────────────┼─────────┤
│ 101 │ 1 │ Ana │ ← Cliente 1 existe en ambas
│ 102 │ 2 │ Luis │ ← Cliente 2 existe en ambas
│ 103 │ 1 │ Ana │ ← Cliente 1 (otra vez)
└─────────┴────────────┴─────────┘

Excluidos:
- Pedido 104 (Cliente 5 no existe en tabla Clientes)
- Cliente 3 (no tiene pedidos)
```

**Cuándo usar:** Cuando SOLO quieres datos que tienen correspondencia en ambas tablas

#### 2. LEFT JOIN (how='left')
**"TODOS del izquierdo, coincidencias del derecho (NaN si no hay)"**

```
Resultado:
┌─────────┬────────────┬─────────┐
│PedidoID │ ClienteID │ Nombre │
├─────────┼────────────┼─────────┤
│ 101 │ 1 │ Ana │ ← Match
│ 102 │ 2 │ Luis │ ← Match
│ 103 │ 1 │ Ana │ ← Match
│ 104 │ 5 │ NaN │ ← Cliente 5 no existe
└─────────┴────────────┴─────────┘

Incluye TODOS los pedidos (tabla izquierda)
Cliente 3 NO aparece (no está en la tabla izquierda)
```

**Cuándo usar:** Cuando quieres TODOS los registros de la tabla principal (izquierda) y añadir info de la otra tabla si existe

**Ejemplo real:** Todos los pedidos, con info de cliente si existe

#### 3. RIGHT JOIN (how='right')
**"TODOS del derecho, coincidencias del izquierdo (NaN si no hay)"**

```
Resultado:
┌─────────┬────────────┬─────────┐
│PedidoID │ ClienteID │ Nombre │
├─────────┼────────────┼─────────┤
│ 101 │ 1 │ Ana │ ← Match
│ 103 │ 1 │ Ana │ ← Match (otra vez)
│ 102 │ 2 │ Luis │ ← Match
│ NaN │ 3 │ Carlos │ ← Cliente 3 sin pedidos
└─────────┴────────────┴─────────┘

Incluye TODOS los clientes (tabla derecha)
Pedido 104 NO aparece (Cliente 5 no existe en tabla derecha)
```

**Cuándo usar:** Menos común. Equivalente a LEFT JOIN cambiando el orden de las tablas

**Ejemplo real:** Todos los clientes, con sus pedidos si existen

#### 4. OUTER JOIN (how='outer')
**"TODO: ambas tablas completas, NaN donde no hay match"**

```
Resultado:
┌─────────┬────────────┬─────────┐
│PedidoID │ ClienteID │ Nombre │
├─────────┼────────────┼─────────┤
│ 101 │ 1 │ Ana │ ← Match
│ 102 │ 2 │ Luis │ ← Match
│ 103 │ 1 │ Ana │ ← Match
│ 104 │ 5 │ NaN │ ← Cliente 5 no existe
│ NaN │ 3 │ Carlos │ ← Cliente 3 sin pedidos
└─────────┴────────────┴─────────┘

Incluye TODO: todos los pedidos Y todos los clientes
NaN donde no hay correspondencia
```

**Cuándo usar:** Cuando necesitas el conjunto completo de datos y quieres identificar dónde faltan relaciones

**Ejemplo real:** Auditoría completa - encontrar pedidos sin cliente O clientes sin pedidos

### Diagrama Resumen Visual

```
INNER: Solo intersección
         ┌──────┐
         │ ∩ │
         └──────┘

LEFT: Todo izquierda + intersección
         ┌──────┬──────┐
         │ L │ ∩ │
         └──────┴──────┘

RIGHT: Intersección + todo derecha
         ┌──────┬──────┐
         │ ∩ │ R │
         └──────┴──────┘

OUTER: TODO (unión completa)
         ┌──────┬──────┬──────┐
         │ L │ ∩ │ R │
         └──────┴──────┴──────┘
```

### Tabla Comparativa

| Join Type | Filas resultado | NaN posibles | Uso típico |
|-----------|----------------|--------------|------------|
| **INNER** | Solo matches | NO | Análisis solo con datos completos |
| **LEFT** | Todos izquierda | En columnas derechas | Enriquecer tabla principal |
| **RIGHT** | Todos derecha | En columnas izquierdas | Raro (usa LEFT cambiando orden) |
| **OUTER** | Todos de ambos | En ambos lados | Análisis completo, auditoría |

### merge(): Combinar por Columnas Comunes

In [ ]:
# Crear DataFrames de ejemplo
df_clientes = pd.DataFrame({
    'ClienteID': [1, 2, 3, 4],
    'Nombre': ['Ana García', 'Luis Pérez', 'Carlos Ruiz', 'María López'],
    'Ciudad': ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
})

df_pedidos = pd.DataFrame({
    'PedidoID': [101, 102, 103, 104, 105],
    'ClienteID': [1, 2, 1, 3, 5], # Cliente 5 no existe en df_clientes
    'Producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Laptop'],
    'Total': [899, 25, 75, 299, 899]
})

print("Tabla de Clientes:")
print(df_clientes)
print()
print("Tabla de Pedidos:")
print(df_pedidos)
print()

In [ ]:
# Inner Join (default): Solo pedidos de clientes existentes
pedidos_con_cliente = pd.merge(df_pedidos, df_clientes, on='ClienteID', how='inner')

print("Inner Join (solo coincidencias):")
print(pedidos_con_cliente)
print()
print("Nota: Pedido 105 (ClienteID=5) fue excluido porque el cliente no existe.")

In [ ]:
# Left Join: Todos los pedidos, info de cliente si existe
todos_pedidos = pd.merge(df_pedidos, df_clientes, on='ClienteID', how='left')

print("Left Join (todos los pedidos):")
print(todos_pedidos)
print()
print("Nota: Pedido 105 tiene NaN en Nombre y Ciudad porque el cliente no existe.")

In [ ]:
# Right Join: Todos los clientes, pedidos si existen
todos_clientes = pd.merge(df_pedidos, df_clientes, on='ClienteID', how='right')

print("Right Join (todos los clientes):")
print(todos_clientes)
print()
print("Nota: María López (ClienteID=4) aparece con NaN en columnas de pedido.")

In [ ]:
# Outer Join: TODO (clientes y pedidos)
union_completa = pd.merge(df_pedidos, df_clientes, on='ClienteID', how='outer')

print("Outer Join (unión completa):")
print(union_completa)
print()
print("Nota: Incluye cliente sin pedidos (4) y pedido sin cliente (5).")

### Merge con Columnas de Nombres Diferentes

In [ ]:
# DataFrames con columnas de nombres diferentes
df_productos = pd.DataFrame({
    'ProductoNombre': ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'Categoría': ['Electrónica', 'Accesorios', 'Accesorios', 'Electrónica'],
    'Stock': [15, 120, 45, 30]
})

# Merge especificando columnas diferentes
pedidos_con_stock = pd.merge(
    df_pedidos,
    df_productos,
    left_on='Producto', # Columna en df_pedidos
    right_on='ProductoNombre', # Columna en df_productos
    how='left'
)

print("Merge con nombres de columna diferentes:")
print(pedidos_con_stock)
print()

# Limpiar columna duplicada
pedidos_con_stock = pedidos_con_stock.drop('ProductoNombre', axis=1)
print("Después de eliminar columna duplicada:")
print(pedidos_con_stock)

### concat(): Concatenar DataFrames

In [ ]:
# Concatenar verticalmente (apilar filas)
ventas_enero = pd.DataFrame({
    'Fecha': ['2024-01-15', '2024-01-20'],
    'Producto': ['Laptop', 'Mouse'],
    'Total': [899, 25]
})

ventas_febrero = pd.DataFrame({
    'Fecha': ['2024-02-10', '2024-02-25'],
    'Producto': ['Teclado', 'Monitor'],
    'Total': [75, 299]
})

ventas_q1 = pd.concat([ventas_enero, ventas_febrero], ignore_index=True)

print("Concatenación vertical (apilar):")
print(ventas_q1)
print()

In [ ]:
# Concatenar horizontalmente (añadir columnas)
df_info = pd.DataFrame({
    'Región': ['Norte', 'Sur', 'Este', 'Oeste']
})

df_ventas_info = pd.DataFrame({
    'Vendedor': ['Ana', 'Luis', 'Carlos', 'María']
})

df_combinado = pd.concat([df_info, df_ventas_info], axis=1)

print("Concatenación horizontal (añadir columnas):")
print(df_combinado)

## 3. Series Temporales

### ¿Qué son las Series Temporales?

Datos con **marcas de tiempo** (timestamps) que representan observaciones a lo largo del tiempo:

- **Precios de acciones:** Un precio cada minuto/hora/día
- **Datos meteorológicos:** Temperatura cada hora
- **Logs de servidores:** Eventos con timestamp
- **Ventas:** Transacciones con fecha

**Pandas es EXCELENTE para series temporales:** manejo de fechas, resampling, rolling windows, análisis de tendencias.

### Crear y Manipular Fechas

In [ ]:
# Convertir strings a datetime
fechas_str = ['2024-01-15', '2024-02-20', '2024-03-10']
fechas_dt = pd.to_datetime(fechas_str)

print("Fechas como datetime:")
print(fechas_dt)
print(f"Tipo: {type(fechas_dt[0])}")
print()

# Crear rango de fechas
rango_diario = pd.date_range('2024-01-01', periods=10, freq='D')
print("Rango de fechas diarias (10 días):")
print(rango_diario)
print()

# Diferentes frecuencias
rango_mensual = pd.date_range('2024-01-01', periods=6, freq='ME')
print("Rango mensual (6 meses):")
print(rango_mensual)
print()

rango_horario = pd.date_range('2024-01-01 00:00', periods=5, freq='h')
print("Rango horario (5 horas):")
print(rango_horario)

### DataFrame con Índice Temporal

In [ ]:
# Crear serie temporal de ventas
np.random.seed(42)
fechas = pd.date_range('2024-01-01', periods=90, freq='D')

df_temporal = pd.DataFrame({
    'Fecha': fechas,
    'Ventas': np.random.randint(100, 500, 90) + np.sin(np.arange(90) / 7) * 50, # Tendencia semanal
    'Visitas': np.random.randint(500, 2000, 90)
})

# Establecer Fecha como índice
df_temporal.set_index('Fecha', inplace=True)

print("Serie temporal de ventas (primeras 10 filas):")
print(df_temporal.head(10))
print()
print(f"Tipo de índice: {type(df_temporal.index)}")

In [ ]:
# Indexación por fechas
print("Ventas del 15 de enero:")
print(df_temporal.loc['2024-01-15'])
print()

# Slicing por rango de fechas
print("Ventas de la primera semana de enero:")
print(df_temporal.loc['2024-01-01':'2024-01-07'])
print()

# Seleccionar mes completo
print("Ventas promedio de enero:")
enero = df_temporal.loc['2024-01']
print(f"Promedio: {enero['Ventas'].mean():.2f}")

### Resampling: Cambiar la Frecuencia Temporal

In [ ]:
# Resample a frecuencia semanal (suma)
ventas_semanales = df_temporal.resample('W')['Ventas'].sum()

print("Ventas semanales (suma):")
print(ventas_semanales)
print()

# Resample a frecuencia mensual (promedio)
ventas_mensuales = df_temporal.resample('ME')['Ventas'].mean()

print("Ventas mensuales (promedio):")
print(ventas_mensuales)

In [ ]:
# Múltiples agregaciones con resample
resumen_mensual = df_temporal.resample('ME').agg({
    'Ventas': ['sum', 'mean', 'max', 'min'],
    'Visitas': ['sum', 'mean']
})

print("Resumen mensual completo:")
print(resumen_mensual)

### Rolling Windows: Medias Móviles

In [ ]:
# Media móvil de 7 días
df_temporal['Ventas_MA7'] = df_temporal['Ventas'].rolling(window=7).mean()

# Media móvil de 30 días
df_temporal['Ventas_MA30'] = df_temporal['Ventas'].rolling(window=30).mean()

print("Serie temporal con medias móviles:")
print(df_temporal[['Ventas', 'Ventas_MA7', 'Ventas_MA30']].tail(10))

In [ ]:
# Visualizar serie temporal con medias móviles
plt.figure(figsize=(14, 6))

plt.plot(df_temporal.index, df_temporal['Ventas'], label='Ventas Diarias', alpha=0.5, linewidth=1)
plt.plot(df_temporal.index, df_temporal['Ventas_MA7'], label='Media Móvil 7 días', linewidth=2)
plt.plot(df_temporal.index, df_temporal['Ventas_MA30'], label='Media Móvil 30 días', linewidth=2)

plt.xlabel('Fecha')
plt.ylabel('Ventas')
plt.title('Serie Temporal de Ventas con Medias Móviles')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Las medias móviles suavizan las fluctuaciones y muestran tendencias.")

### Extraer Componentes de Fechas

In [ ]:
# Resetear índice para trabajar con columna Fecha
df_temporal_reset = df_temporal.reset_index()

# Extraer componentes de fecha
df_temporal_reset['Año'] = df_temporal_reset['Fecha'].dt.year
df_temporal_reset['Mes'] = df_temporal_reset['Fecha'].dt.month
df_temporal_reset['Día'] = df_temporal_reset['Fecha'].dt.day
df_temporal_reset['Día_Semana'] = df_temporal_reset['Fecha'].dt.dayofweek # 0=Lunes, 6=Domingo
df_temporal_reset['Nombre_Día'] = df_temporal_reset['Fecha'].dt.day_name()
df_temporal_reset['Nombre_Mes'] = df_temporal_reset['Fecha'].dt.month_name()

print("DataFrame con componentes de fecha extraídos:")
print(df_temporal_reset[['Fecha', 'Ventas', 'Año', 'Mes', 'Día', 'Nombre_Día']].head(10))

In [ ]:
# Analizar ventas por día de la semana
ventas_por_dia_semana = df_temporal_reset.groupby('Nombre_Día')['Ventas'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)

print("Ventas promedio por día de la semana:")
print(ventas_por_dia_semana)
print()

# Visualizar
plt.figure(figsize=(10, 5))
ventas_por_dia_semana.plot(kind='bar', color='skyblue', edgecolor='black')
plt.xlabel('Día de la Semana')
plt.ylabel('Ventas Promedio')
plt.title('Ventas Promedio por Día de la Semana')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. Pivot Tables y Crosstabs

### Pivot Tables: Reorganizar y Resumir Datos

Las **pivot tables** (tablas dinámicas) son una de las herramientas más poderosas para análisis de datos. Reorganizan datos de formato largo a formato ancho, aplicando agregaciones.

**Analogía:** Como las tablas dinámicas de Excel, pero programáticas y más flexibles.

In [ ]:
# Crear dataset de ventas
df_ventas_pivot = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=20, freq='D'),
    'Región': ['Norte', 'Sur', 'Este', 'Oeste'] * 5,
    'Producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Laptop'] * 4,
    'Ventas': np.random.randint(100, 1000, 20)
})

df_ventas_pivot['Mes'] = df_ventas_pivot['Fecha'].dt.month_name()

print("Dataset de ventas (formato largo):")
print(df_ventas_pivot.head(10))
print()

In [ ]:
# Pivot simple: Ventas por Región y Producto
pivot_region_producto = df_ventas_pivot.pivot_table(
    values='Ventas',
    index='Región',
    columns='Producto',
    aggfunc='sum',
    fill_value=0 # Rellenar NaN con 0
)

print("Pivot Table: Ventas por Región (filas) y Producto (columnas):")
print(pivot_region_producto)
print()

In [ ]:
# Pivot con totales (margins)
pivot_con_totales = df_ventas_pivot.pivot_table(
    values='Ventas',
    index='Región',
    columns='Producto',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print("Pivot Table con totales:")
print(pivot_con_totales)

In [ ]:
# Múltiples valores de agregación
pivot_multiple = df_ventas_pivot.pivot_table(
    values='Ventas',
    index='Región',
    columns='Producto',
    aggfunc=['sum', 'mean', 'count'],
    fill_value=0
)

print("Pivot Table con múltiples agregaciones:")
print(pivot_multiple)

### Crosstab: Tablas de Frecuencias

In [ ]:
# Crosstab: Contar ocurrencias
crosstab_region_producto = pd.crosstab(
    df_ventas_pivot['Región'],
    df_ventas_pivot['Producto'],
    margins=True
)

print("Crosstab: Frecuencias de Región × Producto:")
print(crosstab_region_producto)
print()

# Crosstab con valores (como pivot_table)
crosstab_con_ventas = pd.crosstab(
    df_ventas_pivot['Región'],
    df_ventas_pivot['Producto'],
    values=df_ventas_pivot['Ventas'],
    aggfunc='sum',
    margins=True
)

print("Crosstab con suma de ventas:")
print(crosstab_con_ventas)

## 5. Visualización con Pandas

### Pandas + Matplotlib Integrado

Pandas tiene métodos `.plot()` integrados que usan matplotlib por debajo. Son perfectos para **visualización rápida** durante análisis exploratorio.

In [ ]:
# Gráfico de línea
df_temporal[['Ventas', 'Ventas_MA7']].plot(
    figsize=(12, 5),
    title='Ventas Diarias con Media Móvil',
    xlabel='Fecha',
    ylabel='Ventas',
    grid=True
)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras
ventas_por_producto = df_ventas_pivot.groupby('Producto')['Ventas'].sum().sort_values(ascending=False)

ventas_por_producto.plot(
    kind='bar',
    figsize=(10, 5),
    title='Ventas Totales por Producto',
    xlabel='Producto',
    ylabel='Ventas',
    color='teal',
    edgecolor='black'
)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Histograma
df_temporal['Ventas'].plot(
    kind='hist',
    bins=20,
    figsize=(10, 5),
    title='Distribución de Ventas Diarias',
    xlabel='Ventas',
    ylabel='Frecuencia',
    color='salmon',
    edgecolor='black'
)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot (diagrama de caja)
df_ventas_pivot.boxplot(
    column='Ventas',
    by='Región',
    figsize=(10, 5),
    grid=True
)
plt.suptitle('') # Quitar título automático
plt.title('Distribución de Ventas por Región')
plt.xlabel('Región')
plt.ylabel('Ventas')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot
df_temporal.plot(
    kind='scatter',
    x='Ventas',
    y='Visitas',
    figsize=(10, 6),
    title='Relación entre Ventas y Visitas',
    xlabel='Ventas',
    ylabel='Visitas',
    alpha=0.6,
    s=50,
    c='purple'
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Caso de Uso Completo: Análisis de E-commerce

### Escenario

Tienes datos de un e-commerce con 3 tablas:
1. **Clientes:** ID, Nombre, Ciudad, Fecha_Registro
2. **Pedidos:** PedidoID, ClienteID, ProductoID, Fecha, Cantidad
3. **Productos:** ProductoID, Nombre, Categoría, Precio

**Objetivo:** Análisis completo de ventas, tendencias y clientes.

### Crear Datos Sintéticos

In [ ]:
# Tabla Clientes
np.random.seed(42)

df_clientes_ecom = pd.DataFrame({
    'ClienteID': range(1, 51),
    'Nombre': [f'Cliente_{i}' for i in range(1, 51)],
    'Ciudad': np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], 50),
    'Fecha_Registro': pd.date_range('2023-01-01', periods=50, freq='7D')
})

# Tabla Productos
df_productos_ecom = pd.DataFrame({
    'ProductoID': range(1, 11),
    'NombreProducto': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares',
                       'Webcam', 'SSD', 'RAM', 'Router', 'Cable'],
    'Categoría': ['Electrónica', 'Accesorios', 'Accesorios', 'Electrónica', 'Accesorios',
                  'Accesorios', 'Componentes', 'Componentes', 'Redes', 'Accesorios'],
    'Precio': [899, 25, 75, 299, 59, 89, 120, 85, 49, 15]
})

# Tabla Pedidos (300 pedidos)
n_pedidos = 300
df_pedidos_ecom = pd.DataFrame({
    'PedidoID': range(1001, 1001 + n_pedidos),
    'ClienteID': np.random.randint(1, 51, n_pedidos),
    'ProductoID': np.random.randint(1, 11, n_pedidos),
    'Fecha': pd.date_range('2024-01-01', periods=n_pedidos, freq='8h'),
    'Cantidad': np.random.randint(1, 5, n_pedidos)
})

print(f"Datos creados:")
print(f" - {len(df_clientes_ecom)} clientes")
print(f" - {len(df_productos_ecom)} productos")
print(f" - {len(df_pedidos_ecom)} pedidos")
print()
print("Muestra de Pedidos:")
print(df_pedidos_ecom.head())

### Paso 1: Combinar Datos (Merge)

In [ ]:
# Merge Pedidos + Productos
df_pedidos_full = pd.merge(
    df_pedidos_ecom,
    df_productos_ecom,
    on='ProductoID',
    how='left'
)

# Calcular Total
df_pedidos_full['Total'] = df_pedidos_full['Cantidad'] * df_pedidos_full['Precio']

# Merge con Clientes
df_completo = pd.merge(
    df_pedidos_full,
    df_clientes_ecom[['ClienteID', 'Nombre', 'Ciudad']],
    on='ClienteID',
    how='left'
)

print("Dataset completo (primeras 10 filas):")
print(df_completo.head(10))
print()
print(f"Columnas: {df_completo.columns.tolist()}")

### Paso 2: Análisis con GroupBy

In [ ]:
# Top 5 productos por ventas totales
top_productos = df_completo.groupby('NombreProducto')['Total'].sum().sort_values(ascending=False).head(5)

print("Top 5 Productos por Ventas:")
print(top_productos)
print()

# Top 5 clientes
top_clientes = df_completo.groupby('Nombre')['Total'].sum().sort_values(ascending=False).head(5)

print("Top 5 Clientes por Gasto:")
print(top_clientes)
print()

# Ventas por ciudad
ventas_ciudad = df_completo.groupby('Ciudad')['Total'].agg(['sum', 'mean', 'count'])
ventas_ciudad.columns = ['Total_Ventas', 'Ticket_Promedio', 'Num_Pedidos']

print("Ventas por Ciudad:")
print(ventas_ciudad.sort_values('Total_Ventas', ascending=False))

### Paso 3: Análisis Temporal

In [ ]:
# Establecer Fecha como índice
df_temporal_ecom = df_completo.set_index('Fecha')

# Ventas diarias
ventas_diarias = df_temporal_ecom.resample('D')['Total'].sum()

# Media móvil 7 días
ventas_diarias_ma = ventas_diarias.rolling(window=7).mean()

# Visualizar
plt.figure(figsize=(14, 6))
plt.plot(ventas_diarias.index, ventas_diarias.values, label='Ventas Diarias', alpha=0.5)
plt.plot(ventas_diarias_ma.index, ventas_diarias_ma.values, label='Media Móvil 7 días', linewidth=2)
plt.xlabel('Fecha')
plt.ylabel('Ventas Totales (€)')
plt.title('Evolución de Ventas Diarias')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Paso 4: Pivot Table para Análisis Multidimensional

In [ ]:
# Ventas por Categoría y Ciudad
pivot_categoria_ciudad = df_completo.pivot_table(
    values='Total',
    index='Categoría',
    columns='Ciudad',
    aggfunc='sum',
    fill_value=0,
    margins=True
)

print("Pivot: Ventas por Categoría × Ciudad:")
print(pivot_categoria_ciudad)
print()

# Un mapa de calor de una tabla pivote es, literalmente, dibujar la matriz.
# Aqui se hace con matplotlib a pelo: la visualizacion se estudia en la UD4
# y no merece la pena arrastrar una biblioteca mas solo para esto.
tabla = pivot_categoria_ciudad.iloc[:-1, :-1]  # sin la fila y columna de totales
fig, ax = plt.subplots(figsize=(10, 6))
imagen = ax.imshow(tabla.to_numpy(dtype=float), aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(tabla.shape[1]), tabla.columns, rotation=45, ha="right")
ax.set_yticks(range(tabla.shape[0]), tabla.index)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        ax.text(j, i, f"{tabla.iat[i, j]:,.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagen, ax=ax, shrink=0.8)
ax.set_title("Ventas por categoria y ciudad")
fig.tight_layout()
plt.show()


### Paso 5: Dashboard Resumido

In [ ]:
print("=" * 60)
print(" DASHBOARD DE VENTAS - E-COMMERCE")
print("=" * 60)
print()

# KPIs principales
total_ventas = df_completo['Total'].sum()
num_pedidos = len(df_completo)
ticket_promedio = df_completo['Total'].mean()
num_clientes_unicos = df_completo['ClienteID'].nunique()

print(f" KPIs Principales:")
print(f" • Total Ventas: {total_ventas:,.2f} €")
print(f" • Número de Pedidos: {num_pedidos:,}")
print(f" • Ticket Promedio: {ticket_promedio:.2f} €")
print(f" • Clientes Únicos: {num_clientes_unicos}")
print()

# Top productos
print(f" Top 3 Productos:")
for i, (producto, ventas) in enumerate(top_productos.head(3).items(), 1):
    print(f" {i}. {producto}: {ventas:,.2f} €")
print()

# Top clientes
print(f" Top 3 Clientes:")
for i, (cliente, gasto) in enumerate(top_clientes.head(3).items(), 1):
    print(f" {i}. {cliente}: {gasto:,.2f} €")
print()

# Ciudad con más ventas
ciudad_top = ventas_ciudad['Total_Ventas'].idxmax()
print(f" Ciudad Líder: {ciudad_top} ({ventas_ciudad.loc[ciudad_top, 'Total_Ventas']:,.2f} €)")
print()

# Categoría más vendida
categoria_top = df_completo.groupby('Categoría')['Total'].sum().idxmax()
print(f" Categoría Líder: {categoria_top}")
print()

print("=" * 60)

## Ejercicios Prácticos

### Instrucciones

Completa los siguientes ejercicios para consolidar tu comprensión de Pandas avanzado:

- **Básicos (1-3):** GroupBy y agregaciones
- **Intermedios (4-7):** Merge, series temporales
- **Avanzados (8-10):** Análisis completo integrado

### Ejercicio 1 (Básico): GroupBy Simple

Usa el dataset `df_completo` del caso de uso:
1. Calcula las ventas totales por categoría
2. Encuentra la categoría con mayor número de pedidos
3. Calcula el ticket promedio por categoría

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 2 (Básico): GroupBy Múltiple

Agrupa por Categoría Y Ciudad:
1. Calcula ventas totales para cada combinación
2. Encuentra qué combinación tiene el ticket promedio más alto
3. Cuenta cuántos pedidos hay en cada combinación

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 3 (Básico): Agregaciones Personalizadas

Crea una función que calcule el rango (max - min) y aplícala:
1. Calcula el rango de precios por categoría
2. Usa `.agg()` para obtener sum, mean, min, max, rango en un solo comando

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 4 (Intermedio): Merge Practice

Crea dos DataFrames:
- `df_pedidos_nuevos`: PedidoID, ClienteID, Total
- `df_descuentos`: ClienteID, Descuento_Porcentaje

Combina ambos y calcula el total con descuento aplicado.

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 5 (Intermedio): Series Temporales - Resampling

Usa `df_completo` con índice temporal:
1. Resample a frecuencia semanal (ventas totales)
2. Resample a frecuencia mensual (ventas promedio)
3. Encuentra la semana con mayores ventas

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 6 (Intermedio): Rolling Windows

Calcula:
1. Media móvil de 14 días de ventas
2. Desviación estándar móvil de 7 días
3. Visualiza ambas junto con las ventas diarias

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 7 (Intermedio): Pivot Tables

Crea un pivot table:
- Filas: Producto
- Columnas: Ciudad
- Valores: Número de pedidos (count)
- Incluye totales
- Visualiza como heatmap

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 8 (Avanzado): Análisis de Cohortes de Clientes

**Contexto de Negocio:**

Eres el Data Analyst de un e-commerce y el Director de Marketing te pide un análisis de cohortes para entender la calidad de los clientes que se registran cada mes.

**¿Qué es un Análisis de Cohortes?**

Un **cohorte** es un grupo de usuarios que comparten una característica temporal común, típicamente **el mes de registro**.

**Ejemplo:**
```
Cohorte Ene-2023: Todos los clientes que se registraron en enero 2023
Cohorte Feb-2023: Todos los clientes que se registraron en febrero 2023
...
```

**¿Qué queremos medir?**

1. **LTV (Lifetime Value) promedio por cohorte:** ¿Cuánto gasta en promedio cada cliente de cada cohorte?
2. **Número de clientes activos:** Cuántos clientes de cada cohorte han realizado al menos una compra
3. **Ventas totales:** Total generado por cada cohorte
4. **Tendencias:** ¿Está mejorando la calidad de clientes con el tiempo?

**Preguntas de negocio a responder:**

- ¿Qué cohorte tiene el LTV más alto? ¿Por qué?
- ¿Están mejorando o empeorando nuestros clientes con el tiempo?
- ¿Qué cohorte deberíamos intentar replicar?
- ¿Alguna cohorte necesita reactivación?

---

**Tu tarea (SIN CÓDIGO GUIADO):**

Usando los datasets existentes (`df_completo` y `df_clientes_ecom`), realiza un análisis completo de cohortes:

**PASO 1: Preparación de datos**
- Combina los datos necesarios (pedidos + clientes con fecha de registro)
- Crea una columna de cohorte basada en el mes de registro

**PASO 2: Cálculos clave**
- Calcula ventas totales por cohorte
- Cuenta clientes activos (únicos) por cohorte
- Calcula LTV promedio por cohorte (ventas totales / clientes activos)

**PASO 3: Análisis y visualización**
- Crea un DataFrame resumen con todas las métricas
- Genera mínimo 2 visualizaciones (barras, líneas)
- Identifica mejor y peor cohorte

**PASO 4: Insights de negocio**
- Escribe 3-5 insights accionables
- Recomendaciones específicas basadas en datos

**Conceptos útiles:**
- `.dt.to_period('M')` para extraer mes como periodo
- `pd.merge()` para combinar DataFrames
- `.groupby()` con `.nunique()` para contar clientes únicos
- Visualizaciones con `.plot(kind='bar')`

**Entregable esperado:**
- Código completo ejecutable
- Visualizaciones claras
- Interpretación de negocio (no solo números)

In [ ]:
# TODO: Escribe tu código aquí



### Ejercicio 9 (Experto): Análisis de Logs de Aplicación Web

**Contexto Real:**

Eres el Data Engineer/Analyst de una startup tech. Tu aplicación web genera millones de logs diarios con información de eventos de usuarios: logins, clicks, errores, compras, etc.

El CTO te pide un análisis urgente porque:
- Los servidores se están saturando en ciertos horarios
- Hay reportes de errores intermitentes
- Quieren optimizar la conversión de usuarios

**Dataset: Logs de Aplicación (50,000 eventos)**

Ejecuta el siguiente código para generar el dataset sintético realista:

In [ ]:
# Generar logs de aplicación web (50,000 eventos)
np.random.seed(42)

# Fechas: 30 días de logs, cada 2 minutos aprox
start_date = pd.Timestamp('2024-01-01')
n_logs = 50000

# Crear timestamps con patrones realistas (más tráfico en horas laborales)
timestamps = []
current_time = start_date

for i in range(n_logs):
    # Añadir variabilidad: más eventos durante el día (8am-8pm)
    hour = current_time.hour
    if 8 <= hour <= 20:
        # Horario laboral: eventos cada 30-90 segundos
        increment = pd.Timedelta(seconds=np.random.randint(30, 90))
    else:
        # Noche: eventos cada 2-5 minutos
        increment = pd.Timedelta(seconds=np.random.randint(120, 300))
    
    current_time += increment
    timestamps.append(current_time)

# Crear DataFrame de logs
df_logs = pd.DataFrame({
    'timestamp': timestamps,
    'event_type': np.random.choice(
        ['page_view', 'login', 'logout', 'click', 'purchase', 'error', 'api_call'],
        n_logs,
        p=[0.40, 0.10, 0.05, 0.25, 0.08, 0.07, 0.05] # Probabilidades realistas
    ),
    'user_id': np.random.randint(1000, 5000, n_logs), # 4000 usuarios únicos aprox
    'page': np.random.choice(
        ['/home', '/products', '/cart', '/checkout', '/profile', '/search', '/api/data'],
        n_logs,
        p=[0.30, 0.20, 0.15, 0.10, 0.10, 0.10, 0.05]
    ),
    'response_time_ms': np.random.gamma(2, 50, n_logs).astype(int), # Tiempos de respuesta
    'status_code': np.random.choice(
        [200, 201, 400, 404, 500, 503],
        n_logs,
        p=[0.85, 0.05, 0.03, 0.04, 0.02, 0.01] # Mayoría exitosos
    ),
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], n_logs, p=[0.55, 0.35, 0.10]),
    'region': np.random.choice(['EU', 'US', 'ASIA', 'LATAM'], n_logs)
})

# Añadir algunos patrones interesantes:
# 1. Errores correlacionados (servidores se saturan juntos)
error_mask = df_logs['event_type'] == 'error'
df_logs.loc[error_mask, 'response_time_ms'] = np.random.randint(3000, 10000, error_mask.sum())

# 2. Compras solo en páginas específicas
purchase_mask = df_logs['event_type'] == 'purchase'
df_logs.loc[purchase_mask, 'page'] = np.random.choice(['/checkout', '/cart'], purchase_mask.sum())

print("=" * 70)
print(" DATASET DE LOGS DE APLICACIÓN WEB GENERADO")
print("=" * 70)
print(f"Total de eventos: {len(df_logs):,}")
print(f"Período: {df_logs['timestamp'].min()} → {df_logs['timestamp'].max()}")
print(f"Duración: {(df_logs['timestamp'].max() - df_logs['timestamp'].min()).days} días")
print(f"Usuarios únicos: {df_logs['user_id'].nunique():,}")
print()
print("Primeras 10 filas:")
print(df_logs.head(10))
print()
print("Distribución de tipos de eventos:")
print(df_logs['event_type'].value_counts())


---

**TU MISIÓN (Análisis Completamente Autónomo):**

No hay pasos guiados. Tú decides qué análisis hacer, cómo visualizar, qué métricas calcular.

**Preguntas clave de negocio (elige al menos 5):**

1. **Rendimiento del sistema:**
   - ¿Cuál es el tiempo de respuesta promedio por tipo de evento?
   - ¿Qué páginas son más lentas?
   - ¿Hay correlación entre device y response_time?

2. **Patrones temporales:**
   - ¿Cuáles son las horas pico de tráfico?
   - ¿Hay días con más errores?
   - ¿Los fines de semana tienen patrones diferentes?

3. **Análisis de errores:**
   - ¿Cuántos errores hay por día/hora?
   - ¿Qué páginas tienen más errores?
   - ¿Los errores están concentrados en ciertos horarios?

4. **Comportamiento de usuarios:**
   - ¿Cuántos usuarios únicos por día?
   - ¿Cuál es la tasa de conversión (purchase/total events)?
   - ¿Qué región tiene mejor conversión?

5. **Análisis de dispositivos:**
   - ¿Mobile vs Desktop: quién convierte más?
   - ¿Hay diferencias en response_time por device?

6. **Optimización:**
   - ¿En qué horario escalar servidores?
   - ¿Qué páginas necesitan optimización urgente?

---

**Entregable esperado:**

 **Dashboard completo con:**

1. **KPIs principales** (eventos/día, usuarios únicos, tasa error, response time promedio)

2. **Mínimo 5 visualizaciones:**
   - Serie temporal de eventos por hora
   - Heatmap de tráfico por hora del día y día de semana
   - Distribución de response_time (histograma o boxplot)
   - Top páginas por errores
   - Conversión por región/dispositivo

3. **Análisis de series temporales:**
   - Resampling a granularidad adecuada
   - Rolling windows para detectar tendencias
   - Identificación de anomalías (picos de errores)

4. **Insights de negocio (mínimo 5):**
   - Formato: " **Insight:** [Descripción] → **Acción:** [Recomendación]"
   - Basados en datos reales del análisis
   - Accionables para el equipo técnico/negocio

5. **Recomendaciones técnicas:**
   - ¿Cuándo escalar servidores?
   - ¿Qué optimizar primero?
   - ¿Dónde investigar bugs?

---

**Técnicas esperadas (usa las que necesites):**

- ✓ GroupBy con múltiples dimensiones
- ✓ Series temporales con índice datetime
- ✓ Resampling (horario, diario)
- ✓ Rolling windows
- ✓ Pivot tables / Crosstabs
- ✓ Filtrado avanzado (errores, páginas lentas)
- ✓ Extracción de componentes temporales (hora, día semana)
- ✓ Visualizaciones claras y profesionales

**NO se espera:**
- ✗ Machine learning (solo análisis exploratorio)
- ✗ Código perfecto (prioriza insights sobre código limpio)
- ✗ Responder TODAS las preguntas (elige las más relevantes)

**Este ejercicio es ABIERTO:** Demuestra tu capacidad de análisis autónomo como lo harías en un trabajo real.

In [ ]:
# TODO: Escribe tu código aquí
# 1. Genera el dataset ejecutando el código proporcionado
# 2. Realiza análisis exploratorio completamente autónomo
# 3. Crea visualizaciones que cuenten una historia
# 4. Genera insights accionables



### Ejercicio 10 (Experto): Análisis Multi-Dimensional de Cadena Retail

**Contexto de Negocio Real:**

Eres el Head of Data Analytics de una cadena de tiendas retail con presencia nacional. La empresa tiene:
- 50 tiendas físicas en diferentes ciudades
- Vende múltiples categorías de productos
- Operación de 2 años con datos históricos completos

La CEO te convoca a una reunión ejecutiva en 2 días y necesita respuestas basadas en datos para decisiones estratégicas:

 **Decisiones en juego (alto impacto):**
- ¿Cerrar tiendas no rentables? (afecta 200+ empleos)
- ¿En qué categorías invertir el presupuesto de marketing?
- ¿Expandir a nuevas ciudades? ¿Cuáles?
- ¿Qué tiendas necesitan renovación urgente?

---

**Dataset: Transacciones Retail (100,000 registros, 2 años)**

Ejecuta este código para generar el dataset realista:

```python
# Dataset de retail: 100,000 transacciones, 2 años de operación
np.random.seed(42)

# Parámetros
n_transactions = 100000
start_date = pd.Timestamp('2022-01-01')
end_date = pd.Timestamp('2023-12-31')

# Generar fechas (más ventas en fines de semana y diciembre)
dates = []
current_date = start_date

while len(dates) < n_transactions:
    # Más ventas en fines de semana
    is_weekend = current_date.dayofweek >= 5
    # Más ventas en Q4 (temporada navideña)
    is_q4 = current_date.month >= 10
    
    if is_weekend:
        n_daily = np.random.randint(180, 250)
    elif is_q4:
        n_daily = np.random.randint(150, 220)
    else:
        n_daily = np.random.randint(100, 150)
    
    dates.extend([current_date] * min(n_daily, n_transactions - len(dates)))
    current_date += pd.Timedelta(days=1)

# Recortar a exactamente n_transactions
dates = dates[:n_transactions]

# Tiendas con características diferentes
stores_data = {
    'TiendaID': [f'T{i:03d}' for i in range(1, 51)],
    'Ciudad': np.random.choice(
        ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Málaga', 'Zaragoza', 'Murcia'],
        50
    ),
    'Tamaño': np.random.choice(['Pequeña', 'Mediana', 'Grande'], 50, p=[0.3, 0.5, 0.2]),
    'Años_Operacion': np.random.randint(1, 15, 50)
}
df_stores = pd.DataFrame(stores_data)

# Productos y categorías
products_data = {
    'ProductoID': [f'P{i:04d}' for i in range(1, 201)],
    'Categoría': np.random.choice(
        ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Juguetes', 'Alimentación'],
        200
    ),
    'Precio_Base': np.random.gamma(3, 20, 200) + 10, # Precios entre 10-150€ aprox
    'Margen_%': np.random.uniform(15, 45, 200) # Margen entre 15-45%
}
df_products = pd.DataFrame(products_data)

# Transacciones
df_retail = pd.DataFrame({
    'TransaccionID': [f'TX{i:06d}' for i in range(1, n_transactions + 1)],
    'Fecha': dates,
    'TiendaID': np.random.choice(df_stores['TiendaID'].values, n_transactions),
    'ProductoID': np.random.choice(df_products['ProductoID'].values, n_transactions),
    'Cantidad': np.random.choice([1, 2, 3, 4, 5], n_transactions, p=[0.50, 0.25, 0.15, 0.07, 0.03]),
    'Descuento_%': np.random.choice([0, 5, 10, 15, 20], n_transactions, p=[0.60, 0.20, 0.10, 0.07, 0.03])
})

# Merge para crear dataset completo
df_retail = df_retail.merge(df_stores, on='TiendaID', how='left')
df_retail = df_retail.merge(df_products, on='ProductoID', how='left')

# Calcular métricas financieras
df_retail['Precio_Venta'] = df_retail['Precio_Base'] * (1 - df_retail['Descuento_%'] / 100)
df_retail['Ingresos'] = df_retail['Precio_Venta'] * df_retail['Cantidad']
df_retail['Costo'] = df_retail['Precio_Base'] * (1 - df_retail['Margen_%'] / 100) * df_retail['Cantidad']
df_retail['Beneficio'] = df_retail['Ingresos'] - df_retail['Costo']

# Añadir algunos patrones interesantes:
# Tiendas grandes venden más productos de alto valor
grandes_mask = df_retail['Tamaño'] == 'Grande'
df_retail.loc[grandes_mask, 'Cantidad'] = df_retail.loc[grandes_mask, 'Cantidad'] * 1.3

# Electrónica tiene más descuentos
electro_mask = df_retail['Categoría'] == 'Electrónica'
df_retail.loc[electro_mask, 'Descuento_%'] = df_retail.loc[electro_mask, 'Descuento_%'] * 1.5

# Recalcular con ajustes
df_retail['Cantidad'] = df_retail['Cantidad'].astype(int)
df_retail['Ingresos'] = df_retail['Precio_Venta'] * df_retail['Cantidad']
df_retail['Costo'] = df_retail['Precio_Base'] * (1 - df_retail['Margen_%'] / 100) * df_retail['Cantidad']
df_retail['Beneficio'] = df_retail['Ingresos'] - df_retail['Costo']

print("=" * 70)
print(" DATASET DE CADENA RETAIL GENERADO")
print("=" * 70)
print(f"Transacciones: {len(df_retail):,}")
print(f"Período: {df_retail['Fecha'].min().date()} → {df_retail['Fecha'].max().date()}")
print(f"Tiendas: {df_retail['TiendaID'].nunique()}")
print(f"Productos: {df_retail['ProductoID'].nunique()}")
print(f"Ciudades: {df_retail['Ciudad'].nunique()}")
print(f"Categorías: {df_retail['Categoría'].nunique()}")
print()
print(f"Ingresos totales: {df_retail['Ingresos'].sum():,.2f}€")
print(f"Beneficio total: {df_retail['Beneficio'].sum():,.2f}€")
print(f"Margen promedio: {(df_retail['Beneficio'].sum() / df_retail['Ingresos'].sum() * 100):.1f}%")
print()
print("Primeras 10 filas:")
print(df_retail.head(10))
print()
print("Columnas disponibles:")
print(df_retail.columns.tolist())
```

---

**TU MISIÓN (Análisis Ejecutivo Estratégico):**

Prepara un informe ejecutivo completo con análisis multi-dimensional y recomendaciones accionables.

**NO HAY CÓDIGO GUIADO. TÚ ERES EL ANALISTA EXPERTO.**

---

**Preguntas Estratégicas Clave (responde TODAS):**

### 1. **Análisis de Rentabilidad**
- ¿Qué tiendas son las más/menos rentables?
- ¿Hay tiendas operando con pérdidas? (candidatas a cierre)
- ¿El tamaño de tienda correlaciona con rentabilidad?
- ¿Tiendas más antiguas son más rentables?

### 2. **Análisis de Productos y Categorías**
- ¿Qué categorías generan más ingresos? ¿Y más beneficio?
- ¿Alguna categoría tiene margen negativo?
- Top 10 productos por beneficio
- ¿Los descuentos ayudan o perjudican el beneficio?

### 3. **Análisis Geográfico**
- Ranking de ciudades por beneficio
- ¿Qué ciudades tienen mejor margen?
- ¿Dónde expandir? (ciudades con pocas tiendas pero buena performance)

### 4. **Análisis Temporal**
- Tendencia de ingresos y beneficios (mensual/trimestral)
- ¿Hay estacionalidad? (Q4, verano, etc.)
- ¿Crecimiento año a año (2022 vs 2023)?
- Mejor/peor mes del año

### 5. **Análisis de Comportamiento de Compra**
- Ticket promedio por tienda/categoría
- Cantidad promedio por transacción
- ¿Qué influye en el tamaño de compra?

### 6. **Optimización de Descuentos**
- ¿Los descuentos incrementan ventas lo suficiente para compensar pérdida de margen?
- ¿Qué categorías deberían/no deberían tener descuentos?

---

**Entregable Esperado (INFORME EJECUTIVO COMPLETO):**

### **Parte 1: Executive Summary (KPIs principales)**
```
Ejemplo de formato:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   DASHBOARD EJECUTIVO - CADENA RETAIL 2022-2023
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RENDIMIENTO GENERAL:
   Ingresos Totales: XX.XXM€
   Beneficio Neto: XX.XXM€
   Margen Promedio: XX.X%
   Tiendas Operativas: XX
   Transacciones: XXX,XXX

SALUD DEL NEGOCIO:
  ✓ Tiendas rentables: XX (XX%)
   Tiendas en pérdidas: X (X%)
   Crecimiento YoY: XX%
```

### **Parte 2: Visualizaciones (mínimo 8 gráficos)**

Debe incluir:
1. **Rentabilidad por tienda** (identificar top 10 y bottom 10)
2. **Serie temporal** de ingresos/beneficios (tendencia + estacionalidad)
3. **Comparativa 2022 vs 2023** (crecimiento)
4. **Categorías:** Ingresos vs Beneficio (gráfico de barras comparativo)
5. **Heatmap:** Categorías × Ciudad (identificar combinaciones ganadoras)
6. **Distribución** de tamaño de ticket (histograma/boxplot)
7. **Análisis de descuentos:** Impacto en beneficio
8. **Mapa de rentabilidad:** Ciudades (puede ser tabla con color)

### **Parte 3: Insights Estratégicos (mínimo 8)**

Formato requerido:
```
 **INSIGHT 1: [Título descriptivo]**
    Dato: [Estadística específica con números]
    Interpretación: [Qué significa para el negocio]
    Acción: [Recomendación concreta y accionable]
    Impacto estimado: [Alto/Medio/Bajo - con justificación]
```

Ejemplo:
```
 **INSIGHT 1: 5 tiendas operan con pérdidas sistemáticas**
    Dato: T003, T017, T028, T041, T049 tienen beneficio negativo
           acumulado de -125K€ en 2023
    Interpretación: Estas tiendas drenan 3.2% del beneficio total.
           Llevan >18 meses sin mejorar.
    Acción: Cerrar T003 y T017 (pérdidas >50K€). Renovar T028, T041,
           T049 (tienen potencial pero necesitan inversión).
    Impacto: ALTO - Ahorrar 80K€/año + liberar capital para expansión.
```

### **Parte 4: Recomendaciones Ejecutivas (Top 5 prioridades)**

Priorización clara con impacto y urgencia:
```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   TOP 5 RECOMENDACIONES ESTRATÉGICAS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. URGENTE - [Acción]
   Impacto: [Alto/Medio]
   Timeline: [Inmediato/Q1/Q2]
   Inversión requerida: [Importe]

2. IMPORTANTE - [Acción]
   ...

[etc]
```

### **Parte 5: Apéndice Técnico**

- Metodología utilizada
- Limitaciones del análisis
- Supuestos hechos
- Siguientes pasos recomendados

---

**Técnicas OBLIGATORIAS que debes usar:**

- ✓ GroupBy multi-dimensional (ej: Tienda + Categoría + Mes)
- ✓ Pivot tables complejas (ej: Ciudad × Categoría con beneficio)
- ✓ Series temporales (resampling mensual/trimestral)
- ✓ Rolling windows (tendencias suavizadas)
- ✓ Merge de múltiples DataFrames (si decides enriquecer datos)
- ✓ Cálculos de KPIs derivados (margen%, ticket promedio, etc.)
- ✓ Filtrado y ranking (top/bottom performers)
- ✓ Visualizaciones profesionales con títulos, etiquetas, colores significativos

**Evaluación:**

Tu entrega se evaluará por:
1. **Profundidad del análisis** (¿descubriste insights no obvios?)
2. **Calidad de visualizaciones** (¿cuentan una historia clara?)
3. **Accionabilidad** (¿las recomendaciones son implementables?)
4. **Rigor técnico** (¿uso correcto de Pandas avanzado?)
5. **Pensamiento de negocio** (¿entiendes el impacto real?)

**Esto simula un proyecto real de consultoría estratégica.**

In [ ]:
# TODO: Escribe tu código aquí
# Este es el ejercicio más completo y desafiante
# Demuestra todo lo que has aprendido en este notebook

# ESTRUCTURA SUGERIDA (pero TÚ decides):
# 1. Generar dataset
# 2. Exploración inicial (shape, info, describe)
# 3. Análisis de rentabilidad
# 4. Análisis temporal
# 5. Análisis por categorías/ciudades
# 6. Visualizaciones
# 7. Insights y recomendaciones



## Resumen de Conceptos Clave

### GroupBy: Split-Apply-Combine

1. **GroupBy básico:**
   - `df.groupby('columna')['valor'].agg('funcion')`
   - Funciones: sum, mean, count, min, max, std

2. **Múltiples columnas:**
   - `df.groupby(['col1', 'col2'])`
   - Resultado: MultiIndex

3. **Agregaciones avanzadas:**
   - `.agg(['func1', 'func2'])` para múltiples
   - `.agg({'col1': 'sum', 'col2': 'mean'})` diferentes por columna
   - Funciones personalizadas con `.agg(funcion_custom)`

4. **Operaciones especiales:**
   - `.transform()`: devuelve mismo tamaño que original
   - `.filter()`: filtra grupos completos

### Merge y Join

1. **Tipos de join:**
   - `how='inner'`: solo coincidencias
   - `how='left'`: todos del izquierdo
   - `how='right'`: todos del derecho
   - `how='outer'`: todos de ambos

2. **Sintaxis:**
   - `pd.merge(df1, df2, on='columna', how='tipo')`
   - Columnas diferentes: `left_on`, `right_on`

3. **Concatenación:**
   - `pd.concat([df1, df2], axis=0)` vertical
   - `pd.concat([df1, df2], axis=1)` horizontal

### Series Temporales

1. **Trabajar con fechas:**
   - `pd.to_datetime()` para convertir
   - `pd.date_range()` para crear rangos
   - Establecer como índice: `.set_index('fecha')`

2. **Resampling:**
   - `.resample('D')` diario, `'W'` semanal, `'M'` mensual
   - Agregaciones: `.sum()`, `.mean()`, `.count()`

3. **Rolling windows:**
   - `.rolling(window=7).mean()` media móvil
   - Útil para suavizar tendencias

4. **Extraer componentes:**
   - `.dt.year`, `.dt.month`, `.dt.day`
   - `.dt.dayofweek`, `.dt.day_name()`

### Pivot Tables

1. **Sintaxis básica:**
   ```python
   df.pivot_table(
       values='valor',
       index='fila',
       columns='columna',
       aggfunc='sum',
       fill_value=0,
       margins=True
   )
   ```

2. **Crosstab:**
   - `pd.crosstab(df['col1'], df['col2'])` para frecuencias

### Visualización

1. **Métodos .plot():**
   - `.plot()` línea (default)
   - `.plot(kind='bar')` barras
   - `.plot(kind='hist')` histograma
   - `.plot(kind='scatter')` dispersión

2. **Parámetros útiles:**
   - `figsize`, `title`, `xlabel`, `ylabel`
   - `grid=True`, `alpha`, `color`

### Mejores Prácticas

- ✓ Usa GroupBy en lugar de bucles para agregaciones
- ✓ Establece fechas como índice para series temporales
- ✓ Visualiza datos frecuentemente durante análisis
- ✓ Usa pivot tables para reorganizar datos multidimensionales
- ✓ Combina DataFrames con merge en lugar de bucles
- ✗ No uses bucles for cuando GroupBy puede hacerlo
- ✗ No olvides especificar `how=` en merge (default es inner)
- ✗ No ignores valores NaN después de merge

## Recursos Adicionales

### Documentación Oficial
- [Pandas GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Pandas Merge/Join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Pandas Pivot Tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)

### Tutoriales
- [Real Python - Pandas GroupBy](https://realpython.com/pandas-groupby/)
- [Kaggle - Advanced Pandas](https://www.kaggle.com/learn/pandas)

### Datasets para Practicar
- [Kaggle Datasets](https://www.kaggle.com/datasets)
- [UCI ML Repository](https://archive.ics.uci.edu/ml/index.php)
- [Data.gov](https://www.data.gov/)

### Libros
- "Python for Data Analysis" - Wes McKinney (creador de Pandas)
- "Pandas Cookbook" - Theodore Petrou
- "Effective Pandas" - Matt Harrison

---

**Notebook creado para el Curso de Especialización en IA y Big Data**
**Módulo:** Programación de Inteligencia Artificial
**Unidad:** UD3 - NumPy y Pandas
**Fecha:** 2025

---